# Hard tool-call evaluation

Before running:
1. Settings: Accelerator = GPU T4 x2, Internet = On.
2. Add the private Dataset `mas-3b-toolcall-output` to the notebook.
3. Add the `GITHUB_TOKEN` secret.
4. Run all cells. This does not train; it only runs four 57-case evaluations.


In [ ]:
!nvidia-smi
!df -h /kaggle/working /kaggle/temp

In [ ]:
import os
import subprocess
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("GITHUB_TOKEN")
repo = "/kaggle/working/mas"

if not os.path.exists(repo):
    subprocess.run(
        ["git", "clone", f"https://{token}@github.com/starthong/mas.git", repo],
        check=True,
    )
else:
    subprocess.run(["git", "-C", repo, "fetch", "origin", "master"], check=True)
    subprocess.run(["git", "-C", repo, "reset", "--hard", "origin/master"], check=True)

In [ ]:
%cd /kaggle/working/mas
!pip install -q -r requirements.txt -r requirements-kaggle.txt
!python -c "import transformers, peft, bitsandbytes; print('deps ok')"

In [ ]:
import os
from huggingface_hub import snapshot_download

model_dir = "/kaggle/temp/models/Qwen2.5-3B-Instruct"
os.makedirs(model_dir, exist_ok=True)
snapshot_download("Qwen/Qwen2.5-3B-Instruct", local_dir=model_dir)
print(model_dir)

In [ ]:
!ls -lh /kaggle/input/datasets/starthong/mas-3b-toolcall-output

In [ ]:
%cd /kaggle/working/mas
!bash scripts/run_hard_eval.sh

In [ ]:
%cd /kaggle/working/mas
!bash scripts/run_ood_eval.sh

In [ ]:
import glob
import json

for path in sorted(glob.glob("/kaggle/working/mas/results/hard-*.json") + glob.glob("/kaggle/working/mas/results/ood-*.json")):
    summary = json.load(open(path, encoding="utf-8"))["summary"]
    print(path)
    print(json.dumps(summary, ensure_ascii=False, indent=2))


In [ ]:
!cd /kaggle/working/mas && zip -r hard-ood-results.zip results/hard-*.json results/ood-*.json
!ls -lh /kaggle/working/mas/hard-ood-results.zip
